In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

from img_labormarket.data_management. import 

# Setze die Anzeigeoptionen für Pandas
pd.set_option('display.float_format', '{:.3f}'.format)

In [ ]:
# Load data
df = pd.read_excel('../data/data10.xlsx')

In [ ]:
mapping_df = pd.read_excel('../data/ValueCodings.xlsx')
mapping_df['Frage'].fillna(method='ffill',inplace=True)
mapping_df

def add_codings(df, mapping_df, col_name):
    wert_to_label = dict(zip(mapping_df[mapping_df['Frage'] == col_name]['Wert'],
                             mapping_df[mapping_df['Frage'] == col_name]['Label']))
    df[col_name + '_label'] = df[col_name].astype(str).replace(wert_to_label)
    return df

In [ ]:
mapping_df

In [ ]:
test = dict(zip(mapping_df[mapping_df['Frage'] == 'Q1']['Wert'],
                             mapping_df[mapping_df['Frage'] == 'Q1']['Label']))
df['test'] = df['Q1'].replace(test)
test

In [ ]:
def construct_immigrant_dummy(df):
    # Old values
    df['q2_old'] = np.where(((df['Q2_1c1'].notna())|df['Q2_1c1']=='FAIL'), np.where(df['Q2_1c1'] == 36, 0, 2), np.nan)
    df['q3_old'] = np.where(((df['Q3_1c1'].notna())|df['Q3_1c1']=='FAIL'), np.where(df['Q3_1c1'] == 36, 0, 2), np.nan)
    # generating immigrants dummy
    df['new_Q2'] = pd.to_numeric(df['new_Q2'], errors='coerce')
    df['new_Q3'] = pd.to_numeric(df['new_Q3'], errors='coerce')

    df['img'] = np.where((((df['new_Q2']== 2)&(df['new_Q3']==2)))|((df['q2_old']==2)&(df['q3_old']==2)), 1, 0)
    return df   
df = construct_immigrant_dummy(df)
df['img'].value_counts()

### Gender

In [ ]:
df = add_codings(df, mapping_df, 'Q1')
df = add_codings(df, mapping_df, 'Q4_1')
df.groupby('img')[['Q1_label']].value_counts().unstack()

In [ ]:
df.groupby('img')[['Q60','Q61']].mean()

In [ ]:
df.groupby('img')[['Q4_1_label']].value_counts().unstack()

makes the most sense to atleast condition on the Master students. Begs the question why we din't restrict our sample on them (OR) master students in the frist place. Answer: We though twe'd have more than enough from both.

### Clean Wages

In [ ]:
# Remove wages below minimum wage and above 150k oer year
#df = df.loc[(df['Q26r1']<150) & (df['Q26r1']>=18)]
df = df.loc[(df['Q36r1']<150) & (df['Q36r1']>=18)]

df.groupby('img')[['Q4_1_label']].value_counts().unstack()

# Mincer Wage Regression

In [ ]:
df = df.loc[~df['Q4_1_label'].isin(['Diplom','Magister'])]
# GPA
df['GPA'] = df['Q13r1'].str.replace(',', '.').astype(float)
df = df.loc[df['Q1_label']!='Prefer not to say']
res = smf.ols("np.log(Q36r1) ~ img + C(Q1_label, Treatment('Male')) + C(Q4_1_label)", data=df).fit(cov_type='HC1')
res.summary()

In the most standard mincer regression for starting wages, we cannot detect a signficat wage penalty for immigrants, however the difference at 3.4% is sizeable(ish).

In [ ]:
def recode_degrees(df):
    df = df.copy()
    degree_dict = {'Q6r1':'Engineering',
    'Q6r2':'Economics',
    'Q6r3':'Business',
    'Q6r4':'Finance',
    'Q6r5':'Management',
    'Q6r6':'Biology',
    'Q6r7':'Chemistry',
    'Q6r8':'Physics',
    'Q6r9':'Mathematics',
    'Q6r10':'Computer Science',
    'Q6r11':'Psychology',
    'Q6r12':'Education',
    'Q6r13':'Sociology',
    'Q6r14':'Political Science',
    'Q6r15':'Law',
    'Q6r16':'Medical Studies',
    'Q6r17':'Pharmaceutical Studies',
    }

    df.rename(columns=degree_dict, inplace=True)

    # Step 2: Replace 1s with column names, 0s with NaN
    degree_columns = list(degree_dict.values())
    df[degree_columns] = df[degree_columns].apply(lambda col: col.map(lambda x: col.name if x == 1 else np.nan))

    # Step 3: Create 'Fields of Study' column by combining non-null values into a list
    df['StudyField'] = df[degree_columns].apply(lambda row: row.dropna().tolist(), axis=1)

    return df 

# Step 4: Assign clusters based on the list of fields
def assign_cluster(fields):
    stem = {'Engineering', 'Biology', 'Chemistry', 'Physics', 'Mathematics', 'Computer Science','Medical Studies', 'Pharmaceutical Studies'}
    business = {'Business', 'Economics', 'Finance', 'Management'}
    social = {'Education', 'Sociology', 'Political Science', 'Law','Psychology'}

    if any(f in stem for f in fields):
        return 'STEM & Applied Sciences'
    elif any(f in business for f in fields):
        return 'Business & Economics'
    elif any(f in social for f in fields):
        return 'Social Sciences & Law'
    else:
        return 'Other'
    
def assign_new_cluster(fields):
    stem = {'Engineering', 'Biology', 'Chemistry', 'Physics', 'Mathematics', 'Computer Science','Medical Studies', 'Pharmaceutical Studies'}
    business = {'Business', 'Economics', 'Finance', 'Management'}
    social = {'Education', 'Sociology', 'Political Science', 'Law','Psychology'}

    if any(f in business for f in fields):
        return 'Business & Economics'
    elif any(f in stem for f in fields):
        return 'STEM & Applied Sciences'
    elif any(f in social for f in fields):
        return 'Social Sciences & Law'
    else:
        return 'Other'

df = recode_degrees(df)

In [ ]:
df['DegreeField'] = df['StudyField'].apply(assign_cluster)
df['AltDegreeField'] = df['StudyField'].apply(assign_new_cluster)
df.groupby('img')[['DegreeField']].value_counts().unstack(0)

In [ ]:
df = df.loc[~df['Q4_1_label'].isin(['Diplom','Magister'])]
df = df.loc[df['Q1_label']!='Prefer not to say']
res = smf.ols("np.log(Q36r1) ~ img + C(Q1_label, Treatment('Male')) + C(Q4_1_label) + C(DegreeField)*img", data=df).fit(cov_type='HC1')
res.summary()

# Master students

From now onwards our main mode of comparison is with respect to the groups of Master students, so we can compare people with similar qualifications to each other.

In [ ]:
df = df.loc[df['Q4_1_label']=='Master']
df.groupby('img')['DegreeField'].value_counts()

In [ ]:
res = smf.ols("np.log(Q36r1) ~ img + C(Q1_label, Treatment('Male'))+ C(DegreeField)*img", data=df).fit(cov_type='HC1')
res.summary()

In [ ]:
df.groupby(['img','DegreeField'])['Q36r1'].mean().unstack(0)

In [ ]:
fig, ax = plt.subplots(2,sharex=True, sharey=True, figsize=(12,6))

# First Histogram for STEM & Applied Sciences
ax[0].set_title('STEM & Applied Sciences')
ax[0].hist(df.loc[(df['img']==0) & (df['DegreeField']=='STEM & Applied Sciences')]['Q36r1'], 
           bins=10, alpha=0.5, label='Native')
ax[0].hist(df.loc[(df['img']==1) & (df['DegreeField']=='STEM & Applied Sciences')]['Q36r1'], 
           bins=10, alpha=0.5, label='Immigrants')
ax[0].axvline(df.loc[(df['img']==0) & (df['DegreeField']=='STEM & Applied Sciences')]['Q36r1'].mean(), color='blue', linestyle='dashed', linewidth=2)
ax[0].axvline(df.loc[(df['img']==1) & (df['DegreeField']=='STEM & Applied Sciences')]['Q36r1'].mean(), color='orange', linestyle='dashed', linewidth=2)
# Second Histogram for Business & Economics
ax[1].set_title('Business & Economics')
ax[1].hist(df.loc[(df['img']==0) & (df['DegreeField']=='Business & Economics')]['Q36r1'], 
           bins=10, alpha=0.5, label='Native')
ax[1].hist(df.loc[(df['img']==1) & (df['DegreeField']=='Business & Economics')]['Q36r1'], 
           bins=10, alpha=0.5, label='Immigrants')
ax[1].axvline(df.loc[(df['img']==0) & (df['DegreeField']=='Business & Economics')]['Q36r1'].mean(), color='blue', linestyle='dashed', linewidth=2)
ax[1].axvline(df.loc[(df['img']==1) & (df['DegreeField']=='Business & Economics')]['Q36r1'].mean(), color='orange', linestyle='dashed', linewidth=2)
# Add legend (placing it in the last axis)
ax[1].legend()

# Show plot
plt.show()

In [ ]:
def native_img_comparison(df, vars):
    df = df.copy()
    df_img = df[df['img']==1]
    df_nat = df[df['img']==0]

    rslt = pd.DataFrame(columns=['Immigrant Avg', 'Immigrant Std', 'Immigrant N', 'Native Avg', 'Native Std', 'Native N', 'p-value'])

    for var in vars:
        if var not in df.columns:
            raise KeyError(f"Variable '{var}' not found in DataFrame")
        df_img_var = df_img[var].dropna()
        df_nat_var = df_nat[var].dropna()
        img_mean, img_std, img_n = df_img_var.mean(), df_img_var.std(), df_img_var.count()
        nat_mean, nat_std, nat_n = df_nat_var.mean(), df_nat_var.std(), df_nat_var.count()
        _, pval = ttest_ind(df_img_var, df_nat_var, equal_var=False)
        rslt.loc[var] = [img_mean, img_std, img_n, nat_mean, nat_std, nat_n, pval]

    return rslt

mapping = {
    1: 300,
    2: 900,
    3: 1800,
    4: 2950,
    5: 4200,
    6: 5400,
    7: 7500,
    8: 10500,
    9: 15000,
    10: 21000,
    11: 30000,
    12: 40000  # Letzte Kategorie bleibt unverändert
}

# Ersetzen der Werte in der Spalte 'Q45_b' durch die Mittelpunkte
df['RawOutsideOption'] = df['Q45_b'].replace(mapping)
# Bedingte Logik basierend auf den Werten in 'Q1_label'
df['OutsideOption'] = np.where(df['Q45_a'] == 1, df['RawOutsideOption'] ,
                        np.where(df['Q45_a'] == 2, 0,
                                 np.where(df['Q45_a'] == 3, -df['RawOutsideOption'] , np.nan)))



# Beliefs

In [ ]:
df['immediatejob_diff'] = df['Q47'] - df['Q47_1']
wages = df.groupby('img')['Q36r1'].mean()
wages_avg_img = wages[1]
wages_avg_nat = wages[0]
df['perceived_gap'] = df['Q48r1'] - df['Q48_1r1']
df['native_deviation'] = df['Q48r1'] - wages_avg_nat
df['img_deviation'] = df['Q48_1r1'] - wages_avg_img
vars = ['Q44r1','Q45','Q45_a', 'OutsideOption', 'Q47', 'Q47_1','immediatejob_diff', 'Q36r1' ,'Q48r1', 'Q48_1r1','perceived_gap','native_deviation','img_deviation']  # Add the variables you want to test

rslt = native_img_comparison(df, vars)

rename_dict = {
    'Q44r1': 'Q44r1: Now picture yourself in five years. We would like to know how high you expect your annual earnings (before taxes and deductions) be in five years?',
    'Q45':'Q45: How high is the probability (on a 0-100 scale) that you will be in a high/managerial position in five years?',
    'Q45_a': "Q45_a:  Imagine that you were forced to leave your current job and that you had 3 months to find a job at another employer in the same occupation. Do you think that you would find a job that would offer you a higher overall pay(1), the same pay(2) or a lower pay(3)",
    'OutsideOption': 'Q45_b: What do you think: How much more/less would you earn (annually and gross) in that new job?',
    'Q47': 'Q47: What percent of German graduates (on a 0-100 scale) who graduated from [pipe: Q6] do you think were working full-time immediately after graduation?',
    'Q47_1': 'Q47_1: And what percent of INTERNATIONAL (non-German) graduates do you think were working full-time immediately after graduation?',
    'Q36r1':'Actual Starting Wage',
    'Q48r1': 'Q48r1: What do you think, was the average starting gross annual salary (in euros) of natives graduates who began working full-time immediately after graduation in Germany?',
    'Q48_1r1': 'Q48_1r1: And what do you think, was the average starting gross annual salary (in euros) of international graduates who began working full-time immediately after graduation in Germany?'
}
rslt.rename(index=rename_dict, inplace=True)
rslt[['Immigrant Avg', 'Native Avg', 'p-value']]

In [ ]:
df.loc[df['OutsideOption']<40000].groupby('img')['OutsideOption'].hist(alpha=.5,bins=20,legend=True)

They anticipate that they will be earning considerably more money in five years compared to the natives. Additionally, they think it is likelier that they will be in a managerial position. 

Even in their current job, they believe their outside option to be substantially higher than the one of the natives.

Immigrants expect that fewer immigrants have a job immediately after graduation and they also expect that there are differences in the starting wages. 
Natives expect that the starting wages of immigrants are substantially lower than native and this difference in differen

### Immigrants: Own Wages compared to other Immigrants

In [ ]:
df_img = df.loc[df['img']==1]
ttest_ind(df_img['Q36r1'],df_img['Q48_1r1'],equal_var=False)

### Natives: Own Wages compared to other Natives

In [ ]:
df_nat = df.loc[df['img']==0]
ttest_ind(df_nat['Q36r1'],df_nat['Q48r1'],equal_var=False)

## Reasons for discrimination

Problem is the gap is smaller than expected.

### Degree and Immigrant Split

In [ ]:
df.groupby(['img','DegreeField'])[['Q44r1','Q45']].mean().unstack()

The split in general is larger in STEM. It seems they differ more substantially from their peers.

# Institutional Knowledge

### What factor(s) do you believe were most influential in determining your job search outcomes?

In [ ]:
q27 = [f'Q27r{i}' for i in range(1, 10)]
df['q27avg'] = df[q27].mean(axis=1)
devs27 = [f'Q27r{i}_devs' for i in range(1, 10)]
df[devs27] = df[q27].sub(df['q27avg'], axis=0)

rslt = native_img_comparison(df,devs27)
renamed_dict = {
'Q27r1_devs':'Q27r1: Technical qualifications',
'Q27r2_devs':'Q27r2: Communication skills',
'Q27r3_devs':'Q27r3: University grades',
'Q27r4_devs':'Q27r4: Network in the industry',
'Q27r5_devs':'Q27r5: Internships and student jobs',
'Q27r6_devs':'Q27r6: Letter of recommendation',
'Q27r7_devs':'Q27r7: Engagement in voluntary and social activities',
'Q27r8_devs':'Q27r8: Gender',
'Q27r9_devs':'Q27r9: Nationality/ethnicity',
}
rslt = rslt.rename(index=renamed_dict)
rslt[['Immigrant Avg', 'Native Avg', 'p-value']]

So compared to natives immigrants value:

- Communication skills less
- University grades less
- Network in the industry matters more (Whereas natives deemed it to be a lesser point they deem it to be more important)
- Engagement in voluntary and social activities less
- Both belief Gender does not matter (though it does)
- They think ethnicity matters more than natives.

### What would you change if you were to search for a first job after college all over again?

In [ ]:
q29 = [f'Q29r{i}' for i in range(1, 8)]
df['q29avg'] = df[q29].mean(axis=1)
devs29 = [f'Q29r{i}_devs' for i in range(1, 8)]
df[devs29] = df[q29].sub(df['q29avg'], axis=0)

renamed_dict = {
'Q29r1_devs': 'Q29r1: Use different search channels',
'Q29r2_devs': 'Q29r2: Do more internships',
'Q29r3_devs': 'Q29r3: Use a more appropriate and professional CV',
'Q29r4_devs': 'Q29r4: Negotiate salary',
'Q29r5_devs': 'Q29r5: Tailor each application to the specific job',
'Q29r6_devs': 'Q29r6: Prepare more for job interviews',
'Q29r7_devs': 'Q29r7: Obtain more information about the types of jobs and salary ranges available for my qualifications',
}

rslt = native_img_comparison(df, devs29)
rslt = rslt.rename(index=renamed_dict)
rslt[['Immigrant Avg', 'Native Avg', 'p-value']]

Does not look terribly interesting across native and immigrants. Yet, it seems obtaining information (Q29r7) and negotiating (Q29r4) are the biggest items mentioned. 

### Problems

1. Survivorship Bias: We have people who have a job and stayed in Germany. 
2. Different people altogether
3. Immigrants are a selective subset coming to Germany

### Takeaways

- Wages are not significantly different. Slightly higher for natives
- Natives think they earn average compared to their peers whereas participants in our sample think their salary is better than for other immigrants.
- They think i) their outside options and ii) their wage in five years is substantially higher than their native counterparts.

# Job Search 

### How many months did it take you to find your first job from the time you started your job search?

In [ ]:
rslt = native_img_comparison(df, ['Q19r1'])
rslt[['Immigrant Avg', 'Native Avg', 'p-value']]

On the edge of significance

In [ ]:
# Start job search

### How many applications did you send?

1	Less than 5 applications

2	Between 5 and 10 applications

3	Between 10 and 20 applications

4	Between 20 and 50 applications

5	More than 50 applications

In [ ]:
from scipy.stats import mannwhitneyu

# Assuming `df` has a column "Group" (e.g., "Native" vs. "Immigrant") and "Applications_Sent" (coded 1–5)
group1 = df[df['img'] == 1]['Q22'].dropna()
group2 = df[df['img'] == 0]['Q22'].dropna()

# Perform Mann-Whitney U test
stat, p_value = mannwhitneyu(group1, group2, alternative='two-sided')

print(f"Mann-Whitney U Test: U={stat}, p={p_value}")

Highly significant => Immigrant must send way more applications than natives

### Q20: How far from your residence at that time were you looking for jobs?

1	< 20 km

2	< 50 km

3	< 100 km

4	within Germany

5	within the EU

6	all over the world

In [ ]:
df = add_codings(df, mapping_df, 'Q20')
df.groupby('img')['Q20'].value_counts(normalize=True).unstack()

In [ ]:
from scipy.stats import mannwhitneyu

# Assuming `df` has a column "Group" (e.g., "Native" vs. "Immigrant") and "Applications_Sent" (coded 1–5)
group1 = df[df['img'] == 1]['Q20'].dropna()
group2 = df[df['img'] == 0]['Q20'].dropna()

# Perform Mann-Whitney U test
stat, p_value = mannwhitneyu(group1, group2, alternative='two-sided')

print(f"Mann-Whitney U Test: U={stat}, p={p_value}")

Search further away. (But are not as inclined in search between 20 and 100 km as natives). This could be a family thing. 

In [ ]:
df = add_codings(df, mapping_df, 'Q21')
df.groupby('img')['Q21'].value_counts(normalize=True).unstack()

### Q23: Accept first job?

In [ ]:
# Create a 2x2 table
table = pd.crosstab(df['img'], df['Q23'].replace({1: 'Yes', 2: 'No'}))
n = table.sum(axis=1)
table = table.div(n, axis=0)*100
table

In [ ]:
from scipy.stats import fisher_exact

# Perform Fisher’s Exact Test
odds_ratio, p_value = fisher_exact(table)

print(f"Fisher’s Exact Test: OR={odds_ratio}, p={p_value}")

More immigrants accepted the first offer.

### Initial Reservation Wage

In [ ]:
rslt = native_img_comparison(df, ['Q26r1'])
rslt[['Immigrant Avg', 'Native Avg', 'p-value']]

### Need to extend residence permit

In [ ]:
df.groupby('img')['Q24r1'].value_counts(normalize=True).unstack()

### Q24r3: Salary met my expectations

In [ ]:
df.groupby('img')['Q24r3'].value_counts(normalize=True).unstack()

In [ ]:
df.groupby('img')['Q45'].mean()

In [ ]:
res = smf.ols('np.log(Q36r1) ~ Q1 + img + Q19r1 + GPA + C(Q31_1)', data=df).fit(cov_type='HC1')
res.summary()

### How many employees does company of your first job have?

1	Less than 20 employees

2	Between 20 to 50 employees

3	Between 50 to 200 employees

4	Between 200 to 500 employees

5	Between 500 to 1,000 employees

6	Between 1,000 to 5,000 employees

7	Between 5,000 to 10,000 employees

8	More than 10,000

In [ ]:
df.groupby('img')['Q33'].value_counts(normalize=True).unstack()

More likely to work at very large companies but not fully coherent story here.

### Requirement of your first job?

1	No requirement

2	High School Degree

3	Vocational Training

4	Bachelor Degree

5	Master Degree or higher

6	I don’t know

In [ ]:
df.groupby('img')['Q39'].value_counts(normalize=True).unstack(0)

In [ ]:
# Neue Kodierung: 1 = Kein Universitätsabschluss erforderlich, 2 = Universitätsabschluss erforderlich, 3 = Ich weiß es nicht
def recode_q39(value):
    if value in [1, 2, 3]:
        return 0
    elif value in [4, 5]:
        return 1
    elif value == 6:
        return np.nan
    else:
        return np.nan

df['Q39_new'] = df['Q39'].apply(recode_q39)

### How many observations do we have?

In [ ]:
df.groupby(['img'])['Q39_new'].value_counts()

In [ ]:
df.groupby(['img','Q39_new'])["Q36r1"].mean().unstack(0)

Unfortunately, we only have 911 resp. 13 people in these cells because it seems that similar to our SOEP analysis we have large wage differences for jobs where no university degree is required whereas this difference is not as large when a university degree is required.

### Formal language of first job

In [ ]:
df = add_codings(df, mapping_df, 'Q32r2')
df.groupby(['img'])['Q32r2_label'].value_counts(normalize=True).unstack(0)

Immigrants have jobs where they speak English. We cannot say whether this is a preference or they weren't consider in jobs where the primary language is German. 

### Reasons for different labor market outcomes between immigrants and natives

In [ ]:
df.query('img == 1')[[f'Q52r{i}' for i in range(1,10)]].mean()

For immigrants the largest entries are 3, 4 and 9. They refer to:

- 3. Financial pressure

- 4. Language skills

- 8. Network into the industry

5 and 6 refer to application procedures. It seems they don't necessarily expect large difference there.

# Qualifications

In [ ]:
rslt = native_img_comparison(df, ['GPA','Q14r2','Q15'])
rslt[['Immigrant Avg', 'Native Avg', 'p-value']]

They have a lower GPA and are likelier to have studies in English.  

In [ ]:
df = add_codings(df, mapping_df, 'Q15')
df[df['img']==1]['Q15_label'].value_counts(normalize=True)

### What was the formal language of your studies?

In [ ]:
df.groupby('img')['Q14r2'].value_counts(normalize=True).unstack()

They are likelier to have studied in English

# Personality

In [ ]:
rslt = native_img_comparison(df, ['Q60','Q61'])

renamed_dict = {
'Q60': 'Q60: What is you willingenss to take risk in general?',
'Q61': 'Q61: How would you rate your skills compared to other graduates of your degree program? 1-Very much above average- 5-Very much below average',
}
rslt = rslt.rename(index=renamed_dict)

rslt[['Immigrant Avg', 'Native Avg', 'p-value']]

Immigrants are more risk-taking and more confident about their skills than their native counterparts.